In [2]:
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Literal
load_dotenv()
model = ChatOllama(model='qwen2.5:3b')

In [3]:
class review(TypedDict):
    key_themes: Annotated[list[str], "All key themes discussed in the review."]
    summary: Annotated[str, "A brief summary of the review"]
    sentiment: Annotated[Literal['positive', 'negative'], "Sentiment of the review"]

structured_model = model.with_structured_output(review)

In [4]:
result = structured_model.invoke("The hardware is great, but the software feels bloated. There are too many pre-installed apps that I can't remove. Also, the UI looks outdated compared to other brands. Hoping for a software update to fix this.")
print(result)
result['sentiment']

{'key_themes': ['software', 'user_experience'], 'summary': '用户对软件反馈：硬件质量好，但认为软件存在一些问题。他提到有过多预装的应用无法卸载，并且界面看起来比其他品牌落后。希望软件能够更新以解决这些问题。', 'sentiment': 'negative'}


'negative'

In [5]:
from pydantic import BaseModel, Field
class Review(BaseModel):
    key_themes:list[str] = Field(description="A list of all the key themes discussed in the review.")
    summary: str = Field(description="A brief summary of the review.")
    sentiment: Literal["positive", "negative"] = Field(description="Sentiment of the review.")

pydantic_structured_model = model.with_structured_output(Review)
result = pydantic_structured_model.invoke("The hardware is great, but the software feels bloated. There are too many pre-installed apps that I can't remove. Also, the UI looks outdated compared to other brands. Hoping for a software update to fix this.")
print(result)
result.sentiment

key_themes=['Software Experience', 'App Removal', 'User Interface'] summary="The user is dissatisfied with the current state of the software, specifically mentioning issues related to excessive pre-installed apps that can't be removed and an outdated UI. The user hopes for a software update to address these concerns." sentiment='negative'


'negative'

In [6]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
template1 = PromptTemplate(
    template="Write a detailed report on the {topic}",
    input_variables=['topic']
)

template2 = PromptTemplate(
    template="Write a 5 line summary on the following text.\n {text}",
    input_variables=['text']
)

parser = StrOutputParser()
# prompt1 = template1.invoke({'topic':'warm hole'})
# result = model.invoke(prompt1)
# prompt2 = template2.invoke({'text':result})
# final_result = model.invoke(prompt2)
# final_result

chain = template1 | model | parser | template2 | model | parser
result = chain.invoke({'topic':'black hole'})
result

"Here's a 5-line summary of the provided text:\n\nBlack holes are regions with such intense gravity that nothing can escape them, including light. They form from collapsing massive stars or accreting matter. Black holes have an event horizon beyond which no information escapes, and contain singularities at their centers. Astronomers detect them through X-ray emissions, gravitational lensing, or microlensing events. Understanding black holes is crucial for testing general relativity under extreme conditions and advancing astrophysics."

In [ ]:
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
schema = [
    ResponseSchema(name='fact_1', description="Fact 1 about the topic"),
    ResponseSchema(name='fact_2', description="Fact 2 about the topic"),
    ResponseSchema(name='fact_3', description="Fact 3 about the topic")
]
parser = StructuredOutputParser.from_response_schemas(schema)
template = PromptTemplate(
    template="Give 3 fact about {topic} \n {format_instruction}",
    input_variables=['topic'],
    partial_variables={'format_instruction':parser.get_format_instructions()}
)
# prompt = template.invoke({'topic':'wormhole'})
# result = model.invoke(prompt)
# fin_res = parser.parse(result.content)
chain = template | model | parser
result = chain.invoke({'topic':'wormhole'})
result

{'fact_1': 'Wormholes, also known as Einstein-Rosen bridges, are theoretical passages through space-time that connect distant points in the universe.',
 'fact_2': 'The concept of wormholes was first proposed by Albert Einstein and Nathan Rosen in 1935, using general relativity equations.',
 'fact_3': 'Wormholes require exotic matter with negative energy density to stabilize them; current known forms of matter have positive energy density.'}

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
class Person(BaseModel):
    name:str = Field(description="Name of the person.")
    age:int = Field(gt=18, description="Age of the person")
    city: str = Field(description="Name of the city the person belongs to.")

parser = PydanticOutputParser(pydantic_object=Person)
template = PromptTemplate(
    template="Generate the name, age and city of an imaginary {place} person.\n {format_instructions}",
    input_variables=['place'],
    partial_variables={'format_instructions':parser.get_format_instructions}
)

chain = template | model | parser
result = chain.invoke({'place':'Indian'})
result

Person(name='Aarav Sharma', age=24, city='Mumbai')